# 📥 Notebook 1 — Data Collection & Structural Cleaning
**Project:** AI-Driven Citizen Grievance Analysis

**Dataset:** NYC 311 Service Requests + Balanced Non-Complaints


### What this notebook does:
- Loads the balanced CSV from `data/raw/grievance_complaint_noncomplaint_dataset.csv`
- Inspects shape, columns, null values
- Selects relevant columns
- Fixes missing values
- Parses dates & engineers new features
- Saves cleaned data → `data/cleaned/grievance_cleaned.csv`

---

## Setup Paths

In [1]:
# Paths for data collection and storage ─────────────────────
RAW_DATA_PATH = '../data/raw/grievance_complaint_noncomplaint_dataset.csv'
OUTPUT_DIR    = '../data/cleaned/'

## Load the Balanced Dataset

In [2]:
import pandas as pd

# Load the balanced (complaints + non-complaints) dataset
print("📂 Loading balanced dataset...")
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print(f'✅ Balanced dataset loaded!')
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print(f'Complaints: {df["is_complaint"].sum():,}')
print(f'Non-complaints: {(df["is_complaint"] == 0).sum():,}')

df.head(3)

📂 Loading balanced dataset...


✅ Balanced dataset loaded!
Rows    : 24,774
Columns : 54
Complaints: 12,387
Non-complaints: 12,387


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,...,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Garage Lot Name,Ferry Direction,Ferry Terminal Name,Latitude,Longitude,Location,is_complaint
0,32274996,12/26/2015 11:03:56 PM,12/27/2015 01:27:37 AM,NYPD,New York City Police Department,Public Notice,Area Information,Street/Sidewalk,11372.0,37-23 89 STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,40.749784,-73.877675,"(40.7497844354445, -73.87767513527211)",0
1,32252575,12/22/2015 07:43:42 PM,12/22/2015 09:31:10 PM,NYPD,New York City Police Department,Information Request,Area Information,Street/Sidewalk,11230.0,2008 OCEAN AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,40.612728,-73.953948,"(40.61272781816186, -73.95394818682524)",0
2,31768878,10/16/2015 10:42:12 PM,NaN,NYPD,New York City Police Department,Status Check,Community Update,Street/Sidewalk,NaN,9112-9118 SHORE FRONT PKWY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


##  Check Missing Values

In [3]:
null_df = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %'    : (df.isnull().sum() / len(df) * 100).round(2)
})

# Show only columns that HAVE nulls
print(null_df[null_df['Null Count'] > 0].to_string())

                                Null Count  Null %
Closed Date                           4762   19.22
Descriptor                             189    0.76
Location Type                          132    0.53
Incident Zip                          4670   18.85
Incident Address                      2262    9.13
Street Name                           2262    9.13
Cross Street 1                        6146   24.81
Cross Street 2                        7124   28.76
Intersection Street 1                21588   87.14
Intersection Street 2                22562   91.07
Address Type                          4682   18.90
City                                  4670   18.85
Landmark                             24762   99.95
Facility Type                         4744   19.15
Due Date                                 6    0.02
Resolution Action Updated Date        4800   19.38
X Coordinate (State Plane)            4710   19.01
Y Coordinate (State Plane)            4710   19.01
School or Citywide Complaint   

##  Select Only Useful Columns

In [4]:
# We only need these 11 columns for NLP work
NLP_COLS = [
    'Unique Key',
    'Created Date',
    'Closed Date',
    'Agency Name',
    'Complaint Type',        # ← This is our TARGET LABEL for classification
    'Descriptor',            # ← Main complaint text
    'Location Type',
    'Borough',
    'Status',
    'Resolution Description', # ← How it was resolved
    'is_complaint'           # ← NEW: Binary label for balanced classification
]

df_nlp = df[NLP_COLS].copy()
print(f'Reduced to {df_nlp.shape[1]} columns, {df_nlp.shape[0]:,} rows')
df_nlp.head(3)

Reduced to 11 columns, 24,774 rows


,Unique Key,Created Date,Closed Date,Agency Name,Complaint Type,Descriptor,Location Type,Borough,Status,Resolution Description,is_complaint
0,32274996,12/26/2015 11:03:56 PM,12/27/2015 01:27:37 AM,New York City Police Department,Public Notice,Area Information,Street/Sidewalk,QUEENS,Closed,No violations or issues detected.,0
1,32252575,12/22/2015 07:43:42 PM,12/22/2015 09:31:10 PM,New York City Police Department,Information Request,Area Information,Street/Sidewalk,BROOKLYN,Closed,The situation is under control and satisfactory.,0
2,31768878,10/16/2015 10:42:12 PM,NaN,New York City Police Department,Status Check,Community Update,Street/Sidewalk,Unspecified,Assigned,Everything appears to be in good order.,0


## Fix Missing Values

In [5]:
TEXT_COLS = [
    'Descriptor', 'Resolution Description',
    'Complaint Type', 'Agency Name', 'Location Type'
]

for col in TEXT_COLS:
    df_nlp[col] = df_nlp[col].fillna('unknown')

# Drop rows where Complaint Type is missing
# (it is our label — we cannot train without it)
df_nlp = df_nlp[df_nlp['Complaint Type'] != 'unknown']

print(f'Rows after fixing nulls: {len(df_nlp):,}')
print('Remaining nulls:')
print(df_nlp.isnull().sum())

Rows after fixing nulls: 24,774
Remaining nulls:
Unique Key                   0
Created Date                 0
Closed Date               4762
Agency Name                  0
Complaint Type               0
Descriptor                   0
Location Type                0
Borough                      0
Status                       0
Resolution Description       0
is_complaint                 0
dtype: int64


## Parse Dates & Engineer Features

In [6]:
import numpy as np

df_nlp['Created Date'] = pd.to_datetime(df_nlp['Created Date'], format='mixed', errors='coerce')
df_nlp['Closed Date']  = pd.to_datetime(df_nlp['Closed Date'],  errors='coerce')

# How many hours to resolve the complaint?  (Keep as float for math/sorting)
df_nlp['resolution_hours'] = (
    (df_nlp['Closed Date'] - df_nlp['Created Date'])
    .dt.total_seconds() / 3600
)

# What hour of the day was complaint filed?
df_nlp['hour_created'] = df_nlp['Created Date'].dt.hour

# What day of the week?
df_nlp['day_of_week'] = df_nlp['Created Date'].dt.day_name()

# --- DATA TRANSFORMATION: RESOLUTION METRICS & FORMATTING ---
# This section standardizes date/time reporting for the final report, 
# explicitly handling 'Assigned' or 'Open' cases as "In Progress".

# 1. Format Closed Date: Convert NaT (Open/Assigned) to 'In Progress' for readability
df_nlp['closed_date_fmt'] = np.where(
    df_nlp['Closed Date'].isna(),
    'In Progress',
    df_nlp['Closed Date'].dt.strftime('%Y-%m-%d %H:%M:%S')
)

# 2. Format Resolution Hours: Categorize based on duration and completion status
# - If NaN: Ticket is still open ("In Progress")
# - If < 1 hour: Label as fast resolution ("<1 hr")
# - Otherwise: Round to 1 decimal place and append unit(like: 3.5hr)
df_nlp['resolution_hours_fmt'] = np.select(
    [
        df_nlp['resolution_hours'].isna(),
        df_nlp['resolution_hours'] < 1 
    ],
    [
        "In Progress",
        "<1 hr"
    ],
    default=df_nlp['resolution_hours'].round(1).astype(str) + " hrs"
)

# 3. Convert creation timestamp to 12-hour AM/PM format
df_nlp['hour_created_fmt'] = df_nlp['Created Date'].dt.strftime('%I %p')

print('New columns added: closed_date_fmt, resolution_hours, resolution_hours_fmt, hour_created, hour_created_fmt, day_of_week')
df_nlp[['Status','Created Date','Closed Date', 'closed_date_fmt', 'resolution_hours', 'resolution_hours_fmt', 'hour_created', 'hour_created_fmt', 'day_of_week']].head()

New columns added: closed_date_fmt, resolution_hours, resolution_hours_fmt, hour_created, hour_created_fmt, day_of_week


,Status,Created Date,Closed Date,closed_date_fmt,resolution_hours,resolution_hours_fmt,hour_created,hour_created_fmt,day_of_week
0,Closed,2015-12-26 23:03:56,2015-12-27 01:27:37,2015-12-27 01:27:37,2.394722,2.4 hrs,23,11 PM,Saturday
1,Closed,2015-12-22 19:43:42,2015-12-22 21:31:10,2015-12-22 21:31:10,1.791111,1.8 hrs,19,07 PM,Tuesday
2,Assigned,2015-10-16 22:42:12,NaT,In Progress,NaN,In Progress,22,10 PM,Friday
3,Closed,2015-12-24 23:25:08,2015-12-25 00:15:34,2015-12-25 00:15:34,0.840556,<1 hr,23,11 PM,Thursday
4,Closed,2015-12-22 16:25:42,2015-12-22 19:02:20,2015-12-22 19:02:20,2.610556,2.6 hrs,16,04 PM,Tuesday


##  Create Combined Text Column

In [7]:
# Merge complaint type + descriptor + resolution into one text field
# This gives NLP models more signal to work with
df_nlp['combined_text'] = (
    df_nlp['Complaint Type'].astype(str)         + ' ' +
    df_nlp['Descriptor'].astype(str)             + ' ' +
    df_nlp['Resolution Description'].astype(str)
)

print('Sample combined_text:')
print(df_nlp['combined_text'].iloc[0])

Sample combined_text:
Public Notice Area Information No violations or issues detected.


##  Quick Summary & Save

In [8]:
print('=== FINAL CLEANED DATASET SUMMARY ===')
print(f'Total rows  & columns  : {df_nlp.shape}')
print(f'Unique complaint types (labels): {df_nlp["Complaint Type"].nunique()}')
print(f'Boroughs      : {df_nlp["Borough"].unique().tolist()}')
print(f'Date range    : {df_nlp["Created Date"].min()} → {df_nlp["Created Date"].max()}')
print(f'Balanced classes: {df_nlp["is_complaint"].value_counts().to_dict()}')

# Save output for Notebook 2
SAVE_PATH = OUTPUT_DIR + 'grievance_cleaned.csv'
df_nlp.to_csv(SAVE_PATH, index=False)
print(f'\n✅ Saved → {SAVE_PATH}')
print('👉 Now open Notebook 2: 02_text_preprocessing.ipynb')

=== FINAL CLEANED DATASET SUMMARY ===
Total rows  & columns  : (24774, 18)
Unique complaint types (labels): 28
Boroughs      : ['QUEENS', 'BROOKLYN', 'Unspecified', 'STATEN ISLAND', 'BRONX', 'MANHATTAN']
Date range    : 2015-01-01 07:30:10 → 2015-12-31 23:59:45
Balanced classes: {0: 12387, 1: 12387}



✅ Saved → ../data/cleaned/grievance_cleaned.csv
👉 Now open Notebook 2: 02_text_preprocessing.ipynb
